# Welcome to OpenDT tutorials!

**Two core techniques for datacenters:** We live in an increasingly digitalized society, increasingly dependent on datacenters powering our daily-life digital services. To monitor, operate, and better plan ICT infrastructure, throughout the past decades, simulation enabled time- and cost-efficient what-if analysis, on how various changes may influence the infrastructure. Another technique, monitoring, has enabled operators to analyze historical telemetry and, corroborated with simulation, better reason about infrastructure, workload, and compliance with service-level agreements (SLAs).

**Where do these techniques fall short:** _Monitoring_, however, captures only observable telemetry and omits unobservable telemetry, which exists in reality, but cannot be captured due to privacy or monitoring limitations; monitoring is useful to represent (and reconstruct) the past, but cannot tell how the infrastructure would evolve in the future.
_Simulation_, on the other hand, relies on predictive models that cannot capture peculiar behaviour (e.g., hardware temperature affecting performance, firmware updates, aging) and introduce errors as the infrastructure evolves; simulation is useful to represent static infrastructure, set by a user or a pipeline, but fails to capture evolving infrastructure, drifts, and could lead to erroneous what if analysis.

**OpenDT:** We propose digital twinning: leveraging the strengths of monitoring, of simulation, and automation towards SLA-(better)-compliant operation, with a human supervisor in the loop. OpenDT is the first open-source digital twin for monitoring and operating datacenters, and, in these tutorials, is coupled (in-silico) with a cluster from SURF, the Dutch national supercomputing center.

**The workload:** Throughout these notebooks, we use SURF-22, a scientific workload trace used in peer-reviewed experiments, and traced in one of the largest HPC facilities at SURF, the Netherlands. It traces scientific jobs run in production, batch, with an average duration of 39.52 CPU-hours, and also detailed energy use. SURF-22 spans over 7 days, 7,850 jobs, over 310,000 CPU-hours.

**In these tutorials...**
1. _Dynamic autoscaling_ ([1.autoscaling.ipynb](1.autoscaling.ipynb), about 10 minutes): you will learn concepts about monitoring, simulation, and digital twinning. You will learn, hands-on, what _autoscaling_ means for digital twins, how to quantify against ground truth, and how to explore what-if scenarios with fewer or more hosts.
2. _Simulation calibration_ ([2.calibration.ipynb](2.calibration.ipynb), about 20 minutes): you will learn how digital twins can correct themselves. You will explore how various configurations of OpenDT's calibrator can impact the accuracy itself, and where are trade-offs between exploration breadth and efficiency.
3. _CO2-aware workload migration_ ([3.co2_migration.ipynb](3.co2_migration.ipynb), about 10 minutes): you will learn how location influences CO2 emissions (e.g., why a workload deployed in Germany emits 14 tons of CO2, while in Switzerland only 100 kg). Hands-on, you will compare four locations and reason on advantages or disadvantages for each location.

# Notebook 1: Digital twins for datacenter autoscaling

Having a large number of machines can be beneficial for the performance of a datacenter, but idle machines can result in a significant energy cost. In contrast, too few machines can hurt queueing times, and even job feasibility. So... how many machines does a datacenter really need? What is too few and what is too much? And how to find out these numbers?

In this notebook, we explore infrastructure scaling using an in-silico cluster from SURF, and interacting with OpenDT tied to this infrastructure. You will learn:
1. What a digital twin for datacenters is and how OpenDT twins SURF-SARA, a scientific cluster of 277 hosts.
2. How to quantify the accuracy of a digital twin against the ground truth.
3. How up-/down-scaling influences power draw and job-level metrics.
4. How many hosts the SURF cluster needs over a day, and how much energy a dynamic autoscaler could save.


By the end of the notebook, you will be able to validate a digital twin against real-world telemetry, explore what-if scenarios on the infrastructure, and reason, with numbers, about the trade-off between performance and energy when scaling a datacenter up or down.

> Run the cells in order (Shift+Enter). For each question, click your selected option to reveal the answer.

In [ ]:
import pandas as pd
from resources import *

check_setup()

# 1. What is a digital twin for datacenters?

Firstly, we are aligning on terminology used in this and in the following notebooks.


**Monitoring** = collecting telemetry data from the ICT infrastructure, either directly observable metrics (e.g., instantaneous power draw, available devices), or derived information (e.g., job progress, operational phenomena).

**Simulation** = predicting how the ICT infrastructure operates under workload (e.g., performance, energy consumption, CO2 emissions), in a time- and cost-efficient way; OpenDT (digital twin) uses OpenDC (simulator), which is peer-reviewed and discrete-event.

**What-if analysis** = simulating how various changes (e.g., more or fewer hosts, scheduler comparison) may influence the infrastructure under workload.

**Ground truth** = the measured reality (e.g., the measured amount of energy consumed), against which we quantify accuracy. It can also be regarded as an erroneous term, because measurements are not always "ground truth," but it is the community standard term.

**Window of operation** = discretized periods of time in which the digital twin receives telemetry, runs simulations, processes results, makes recommendations for infrastructure steering, then continues to the next window.

**In silico** = run "in-silicium" (in silicon) e.g., a real-world datacenter simulated on the computer's silicon. In these notebooks, we use in-silico digital twins: it is almost free to experiment with a simulated datacenter vs. to run a real-world one.

**In vitro** = real-world, but run in a controlled environment e.g., testing how a feature operates in real-world, but without the influence of external factors.

**In vivo** = real-world, in an uncontrolled environment e.g., real-world deployment of a digital twin with a production cluster.

**Digital twin** (the 6G Future Network Services definition):
_"The digital
twin is a digital representation, possibly multi-model and multiscale, of a part or of the whole of the ICT infrastructure, up to the
entire digital continuum in which it operates. The digital twin can help with one or several of design, analysis, operation, testing,
and benchmarking a part of or the whole infrastructure based on performance, risk, cost, and sustainability requirements. The
digital twin can leverage current and past telemetry from the infrastructure to do short- and/or long-term twinning. The digital
twin’s autonomy is controlled and ranges from full human-supervision to full autonomy with post-factum auditing"_. Figure 1 depicts the high-level architecture of a digital twin for datacenters.

![Datacenter digital twinning](resources/images/digital_twinning.png)

*High-level overview of datacenter digital twinning; a human-in-the-loop (HITL) oversees the process (OpenDT paper,
Figure 1).*

# 2. How to quantify accuracy for digital twins?

We quantify accuracy by comparing measured reality (also referred to as "ground truth") with the simulator's predictions. For this, we firstly need to align, by timestamp, the two measurements. Then, we select a representative metric, such as MAPE (the Mean Absolute Percentage Error), which penalizes over-estimates and under-estimates equally and is a community standard to quantify accuracy.


$\text{MAPE}\,[\%] = \frac{1}{n} \sum_{i=1}^{n} \frac{|R_i - S_i|}{R_i} \times 100$, where
- $n$ is the number of samples,
- $R$ is the real-world data,
- $S$ is the simulation data,
- $i$ is the sample index.

Since MAPE is an error ratio, naturally, the lower the better.

For datacenter simulation, the community-accepted MAPE error ratio is under 10%.

#### Let's quantify OpenDT's MAPE!

In [ ]:
DAY = TRACE_START + pd.Timedelta("24h")  # the first 24 hours of the trace

day1 = simulate(until=DAY, label="OpenDT")
plot_power(day1, hourly_mape=True)
print(f"MAPE: {mape(day1):.1f}%")

In [ ]:
ask_question("1.1")


# 3. What are the implications of upscaling or downscaling infrastructure?

We evaluate Day 1 of the SURF workload. The SURF workload spans over 277 hosts, each host with 128 GB RAM, 16 cores, and 2.1 GHz.

What if we downscale to a quarter (69 hosts) or half (138 hosts), or upscale 10x (2,770 hosts)?

In [ ]:
ask_question("1.2")

In [ ]:
sizes = [69, 138, 277, 2770]
runs = simulate_many("hosts", sizes, until=DAY)

print("""(1) avg_power_kw = average power draw of the entire cluster,
(2) jobs_waited = jobs that waited in the queue,
(3) mean_wait_min = average time in queue [min],
(4) waiting_at_end = jobs waiting at the end of the day (remember, we analyze just day 1 out of 7 days)""")

table = pd.DataFrame([run.summary() for run in runs.values()], index=list(runs))
columns = ["avg_power_kw", "jobs_waited", "mean_wait_min", "waiting_at_end"]
table[columns].style.format(precision=1, thousands=",").format_index(thousands=",")

In [ ]:
ask_question("1.3")

In [ ]:
ask_question("1.4")

In [ ]:
plot_power(*runs.values(), energy=True);

# 4. How many hosts are actually needed, for SURF-SARA, October 2022, day 1 of workload?

In the simulation, each running job occupies one entire host (16 cores): hosts in use = running jobs, every 150 s.
A dynamic autoscaler would keep these hosts on, plus a few spare ones, and switch off the rest.

This is where a digital twin comes into play: dynamic autoscaling, for minor changes (e.g., 1% more/fewer hosts), but with a human-in-the-loop for major changes (e.g., power off 50% of the hosts).

In [ ]:
HOSTS = 277  # hosts at SURF
in_use = day1.service["tasks_active"]  # busy hosts at each moment (one job per host)

hosts = pd.DataFrame({"Hosts in Use": in_use, "Idle Hosts": HOSTS - in_use})
ax = hosts.plot.area(color=["#009E73", "#F0E442"], linewidth=0)
ax.set(ylim=(0, 300), xlabel="Time [UTC]", ylabel="Hosts")
legend_above(ax)

In [ ]:
ask_question("1.5")

In [ ]:
SPARE = 10  # hosts kept on for new jobs
IDLE_POWER = 25  # watts an idle SURF host draws

switched_off = (HOSTS - in_use - SPARE).clip(lower=0)  # idle hosts, minus the spare ones
saved_kwh = switched_off.mean() * IDLE_POWER * 24 / 1000  # hosts off on average x W x 24 h, in kWh
used_kwh = day1.summary()["energy_kwh"]
print(f"Switching off idle hosts saves {saved_kwh:.0f} of {used_kwh:.0f} kWh ({saved_kwh / used_kwh:.0%}).")

In [ ]:
ask_question("1.6")

In [ ]:
ask_question("1.7")

# 5. Live, on OpenDT: Change the number of hosts from 277 to 2,770. What do you observe?

*Optional.* In the live demo, we change the hosts on the dashboard (Datacenter, Hosts 2,770); from then on, OpenDT twins
the datacenter with 2,770 hosts. The same view, from two simulations: prior to the jump, OpenDT twinning 277 hosts, then upscaling to 2,770 hosts, and drifting about 4x from the ground truth:

In [ ]:
SWITCH = TRACE_START + pd.Timedelta("10h")  # 08:00 UTC on 7 October

scaled = day1.power["power_draw"].copy()
scaled[SWITCH:] = runs[2770].power["power_draw"][SWITCH:]  # from 08:00, the simulation with 2,770 hosts
plot_power(scaled.rename("OpenDT, 2,770 hosts from 08:00"));

*Next:* [2.calibration.ipynb](2.calibration.ipynb), simulation calibration. We're diving deeper into the feature of a self-calibrating digital twin: OpenDT.